# Reproduce the SWE-bench gold-coupling census, then ask Gemma 4 the audit question

This notebook goes with the writeup *"Gold-Coupled Tests: Auditing Reference Overfitting in SWE-bench Hardening"* and its dataset
(`swe-bench-gold-coupling-census`).

- **Part 1 (CPU, about 10 seconds, no internet).** Runs `reproduce.py`, which recomputes 156 numbers of the writeup
  from the shipped CSV and JSON files and checks each one, then loads the data with pandas, shows the class shares
  and redraws Figure 1.
- **Part 2 (GPU, optional, off by default).** Loads Gemma 4 E4B-it, the smallest Gemma 4 the paper ran, and asks it
  the audit question on 2 benchmark tasks with the paper's exact prompt and verdict parser. Edit `RUN_GEMMA` to
  `True` in the first code cell to run it; it needs the "GPU T4 x2" accelerator and internet (see Part 2 for why).

"Run All" with the default settings runs Part 1 only.

In [ ]:
# Settings: the only cells you may want to change are this one and the Part 2 settings below.
import os
import tempfile
from pathlib import Path

RUN_GEMMA = os.environ.get("RUN_GEMMA", "0") == "1"   # Part 2 off by default; set to True on a GPU T4 x2 session
KAGGLE_DATASET_DIR = Path("/kaggle/input/swe-bench-gold-coupling-census")


def find_release() -> Path:
    # The attached Kaggle dataset first; then any attached copy; then the repository folder this notebook sits in.
    if (KAGGLE_DATASET_DIR / "reproduce.py").is_file() and (KAGGLE_DATASET_DIR / "data/auditor_benchmark").is_dir():
        return KAGGLE_DATASET_DIR
    if Path("/kaggle/input").is_dir():
        for hit in [h for pat in ("*/reproduce.py", "*/*/reproduce.py", "*/*/*/reproduce.py")
                    for h in sorted(Path("/kaggle/input").glob(pat))]:
            if (hit.parent / "data/auditor_benchmark").is_dir():
                return hit.parent
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "reproduce.py").is_file() and (d / "data/auditor_benchmark").is_dir():
            return d
    raise SystemExit("release not found: attach the dataset or run from the repository's notebooks/ folder")


RELEASE = find_release()
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(tempfile.gettempdir()) / "swe_abs_census_nb"
WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path(tempfile.gettempdir()) / "swe_abs_census_cache"   # fetched third-party text stays out of the outputs
CACHE.mkdir(parents=True, exist_ok=True)
print("release:", RELEASE.name, "| writable output folder:", WORK.name)

## Part 1: reproduce the headline numbers

`reproduce.py` runs as a separate process from the (read-only) dataset folder. It prints one line per check and ends
with `ALL 156 CHECKS PASS`; it exits 1 on any failed check. Bootstrap intervals are byte-equal with numpy 2.5.2; with
another numpy version an interval bound may move in the last digit, and the script then stops with a message saying so.
In that case only, the cell installs numpy 2.5.2 into a temporary folder for the `reproduce.py` process alone and
retries; that fallback needs internet and Python 3.12 or later (numpy 2.5 wheels).

In [ ]:
import subprocess
import sys

import numpy as np


def run_reproduce(extra_path: str | None = None):
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
    if extra_path:
        env["PYTHONPATH"] = extra_path + os.pathsep + env.get("PYTHONPATH", "")
    return subprocess.run([sys.executable, "reproduce.py"], cwd=RELEASE, env=env, capture_output=True, text=True)


res = run_reproduce()
if res.returncode != 0 and np.__version__ != "2.5.2":
    # Only on a numpy mismatch: pin numpy 2.5.2 for the reproduce.py process alone (needs internet), leave the kernel as is.
    last = (res.stdout + res.stderr).strip().splitlines() or ["(no output)"]
    print(f"numpy {np.__version__} gave: {last[-1]}")
    print("retrying with numpy 2.5.2 installed into a separate folder ...")
    target = str(CACHE / "numpy_2_5_2")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", target, "numpy==2.5.2"], check=False)
    res = run_reproduce(target)

lines = res.stdout.strip().splitlines()
(WORK / "reproduce_log.txt").write_text(res.stdout + res.stderr, encoding="utf-8")
ok = [x for x in lines if x.startswith("OK")]
other = [x for x in lines if x.split(" ", 1)[0] not in ("OK", "elapsed", "ALL")]
print(f"{len(ok)} checks print OK; first three:")
print("\n".join(ok[:3]))
print("...")
print("\n".join(other) if other else "(no check printed anything other than OK)")
print("\n".join(lines[-2:]))
if res.returncode != 0:
    print(res.stderr[-2000:])
passed = bool(lines) and (lines[-1] == "ALL 156 CHECKS PASS" or lines[-1].startswith("ALL 156 CHECKS PASS ("))  # may carry a DRAFT-FIX note
assert res.returncode == 0 and passed, "reproduce.py did not pass; see the log above"

### The data in three cells

`entries.csv` has one row per resolved leaderboard entry (system x task) with our re-run labels on the original tests
and on SWE-ABS (`1` pass, `0` fail, empty = no valid verdict). An entry is *judged* when it passes the original tests
and SWE-ABS gives a complete verdict; a judged entry that fails SWE-ABS is *wrong-but-passing* (wbp), a rejection.

In [ ]:
import pandas as pd

entries = pd.read_csv(RELEASE / "data/labels/entries.csv")
verdicts = pd.read_csv(RELEASE / "data/task_verdicts.csv")
judged = entries[(entries.label_official == 1) & entries.label_hardened.notna()]
wbp = judged[judged.label_hardened == 0]
print(f"{len(entries):,} entries from {entries.system.nunique()} systems on {entries.instance_id.nunique()} tasks")
print(f"{len(judged):,} judged; SWE-ABS rejects {len(wbp):,} ({len(wbp) / len(judged):.1%}) in {wbp.instance_id.nunique()} tasks")
print(f"{len(verdicts)} tasks with a rejection, one verdict each")
verdicts[["instance_id", "class", "kind", "first_category", "top_test", "abs_wbp", "abs_judged"]].astype({"kind": "Int64"}).head()

Each of the 305 tasks with a rejection has one verdict on its most-failed SWE-ABS test. A task is **coupled** when a
first reader and an adversarial refuter both say the test checks something the issue does not ask for (kind 1: it
pins an artefact of the gold patch; kind 2: it checks where gold improved on the issue's own proposed fix),
**contested** when the refuter disagrees, **in_scope** when the test checks what the issue asks (a genuine rejection), and **env** when the rejection is an
environment failure, not a test verdict (one task).

In [ ]:
by_entry = wbp.merge(verdicts[["instance_id", "class", "kind"]], on="instance_id", validate="many_to_one")
label = by_entry["class"].where(by_entry["class"] != "coupled",
                                "coupled, kind " + by_entry["kind"].astype("Int64").astype(str))
tasks_label = verdicts["class"].where(verdicts["class"] != "coupled",
                                      "coupled, kind " + verdicts["kind"].astype("Int64").astype(str))
shares = pd.DataFrame({"tasks": tasks_label.value_counts(), "rejected entries": label.value_counts()})
shares["share of rejected entries"] = (shares["rejected entries"] / len(wbp)).map("{:.1%}".format)
order = ["coupled, kind 1", "coupled, kind 2", "contested", "in_scope", "env"]
shares = shares.reindex(order)
print(f"coupled in all: {(verdicts['class'] == 'coupled').sum()} tasks, "
      f"{(by_entry['class'] == 'coupled').mean():.1%} of rejected entries")
shares

Figure 1 is redrawn from `data/system_rates/per_system_corrected.csv` by the shipped script (it checks both medians
against the saved summary before drawing). The figure is written to the output folder, not the dataset.

In [ ]:
import importlib.util

from IPython.display import Image, display

spec = importlib.util.spec_from_file_location("fig1_rates", RELEASE / "figures/fig1_rates.py")
fig1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fig1)
fig1.OUT = WORK                       # the dataset folder is read-only on Kaggle
fig1.main()
display(Image(filename=str(WORK / "fig1_rates.png"), width=900))

## Part 2 (optional, GPU): Gemma 4 E4B audits two benchmark tasks

**What runs.** The paper asks: *does the issue, with its hints, ask for what this hardened test checks?* Part 2 puts
that question to Gemma 4 E4B-it, with the exact prompt and verdict parser of the paper's auditor runs
(`code/audit_prompt.py`), greedy decoding, fp16, and the same model revision when the weights come from Hugging
Face. On the 76 decided benchmark tasks the paper
reports E4B at Cohen's kappa 0.127 [95% CI -0.090, 0.327] against the two-reader reference: weak agreement. Two tasks
show what one audit costs and looks like; they are a demonstration, not an evaluation, and say nothing new about
accuracy.

**Which two tasks.** A fixed rule, not chosen by outcome: among the decided benchmark tasks, the one with the
shortest A1 prompt in each reference class (`coupled`, `genuine`). The saved A1 verdict is printed next to the live one.

**Why internet.** The dataset does not redistribute issue texts, gold patches, SWE-ABS test files or agent patches.
The notebook rebuilds each task's input at run time from public sources: the issue, hints and gold patch from SWE-bench
Verified on Hugging Face, the SWE-ABS test patch from SWE-ABS's Hugging Face dataset, and the agent patches from the
public SWE-bench/experiments submissions. Both Hugging Face datasets are fetched at a pinned revision. Every text
that goes into the prompt is checked before use: the issue, the hints, the gold patch and the SWE-ABS test patch
(its first 6,000 characters, as the paper's prompt used it) against the sha256 values in
`packet_index.jsonl` (`text_sha256`), and each agent patch against its sha256 there; a mismatch stops the notebook.
The model weights come from an attached Kaggle model if there is one, else from Hugging Face at the revision of the
paper's runs (accept the Gemma licence and add an `HF_TOKEN` secret); the loaded path and revision are printed. Inference itself runs locally on the session's GPUs; nothing is sent to an API.

**One honest difference from the paper's runs.** The paper's prompt also listed, for each top failing test, one error
line from our own test runs. Those run logs are not public, so that section reads "(none)" here; the rest of the
prompt is rebuilt from the sources above. The live verdict can therefore differ from the saved A1 verdict.

**Memory.** E4B in fp16 holds about 14.8 GiB of weights, more than one T4 holds (a 16 GB card, about 14.6 GiB usable). As in the
paper's runs, the model is split over Kaggle's two T4s with the per-GPU limits of `configs/audit.json`.

In [ ]:
# Part 2 settings
TASK_RULE = "shortest A1 prompt per reference class among decided tasks"
MODEL_OVERRIDE = os.environ.get("NB_MODEL_ID")   # testing only: a tiny random Gemma 4 for a CPU dry run
if not RUN_GEMMA:
    print("Part 2 is off (RUN_GEMMA = False). Edit RUN_GEMMA to True in the first code cell, on a GPU T4 x2 session, to run it.")

In [ ]:
import hashlib
import io
import json
import time
import urllib.request

sys.path.insert(0, str(RELEASE / "code"))
import audit_prompt  # noqa: E402  the paper's prompt and parser, verbatim


def read_jsonl(p):
    return [json.loads(x) for x in Path(p).read_text(encoding="utf-8").splitlines() if x.strip()]


BENCH = RELEASE / "data/auditor_benchmark"
labels = {r["instance_id"]: r for r in read_jsonl(BENCH / "labels.jsonl")}
index = {r["instance_id"]: r for r in read_jsonl(BENCH / "packet_index.jsonl")}
saved = {(r["model"], r["instance_id"]): r for r in read_jsonl(BENCH / "predictions/a1-r1-e4b-12b.jsonl")}
cfg = json.loads((BENCH / "configs/audit.json").read_text(encoding="utf-8"))
E4B = next(m for m in cfg["models"] if m["name"] == "e4b")

picked = {}
for gold in ("coupled", "genuine"):
    pool = [i for i, r in labels.items() if r["gold"] == gold]
    picked[gold] = min(pool, key=lambda i: (saved[("e4b", i)]["prompt_tokens"], i))
TASKS = [picked["coupled"], picked["genuine"]]
for t in TASKS:
    print(f"{t}: reference {labels[t]['gold']}, A1 prompt {saved[('e4b', t)]['prompt_tokens']} tokens")

In [ ]:
HF = "https://huggingface.co/datasets/{repo}/resolve/{rev}/data/test-00000-of-00001.parquet"
SOURCES = {   # dataset, pinned revision (the revision the census packets match)
    "verified": ("princeton-nlp/SWE-bench_Verified", "c104f840cc67f8b6eec6f759ebc8b2693d585d4a"),
    "abs": ("OpenAgentLab/SWE-Bench_Verified_ABS", "79fc07a8f4ce5e3720e069948073f5f910b7577b"),
}


def http(url: str) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "swe-abs-census-notebook"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read()


def suite(name: str) -> pd.DataFrame:
    repo, rev = SOURCES[name]
    cache = CACHE / f"{name}-{rev[:12]}.parquet"
    if not cache.is_file():
        cache.write_bytes(http(HF.format(repo=repo, rev=rev)))
    return pd.read_parquet(cache).set_index("instance_id")


def agent_patch(iid: str, ref: dict) -> str:
    # The patch an entry submitted, from SWE-bench/experiments, checked against the shipped sha256.
    import fetch_patches as fp            # release/code; caches under fp.DATA/raw: a temporary folder, not the outputs
    fp.DATA = CACHE
    for system in ref["systems"]:
        meta = (fp.get(f"{fp.RAW}/{system}/metadata.yaml") or b"").decode("utf-8", errors="replace")
        base = fp.logs_base(meta)
        text = fp.fetch_patch(base, system, iid) if base else None
        if text and sha(text) == ref["sha256"]:
            return text
    raise SystemExit(f"{iid}: no source returned the patch with sha256 {ref['sha256'][:16]}")


def sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def build_packet(iid: str, verified: pd.DataFrame, abs_suite: pd.DataFrame) -> dict:
    # The fields audit_prompt.render reads, as the paper's packet builder filled them; first_errors is the one gap.
    v, ix = verified.loc[iid], index[iid]
    texts = {"issue": v["problem_statement"], "hints": v["hints_text"] if isinstance(v["hints_text"], str) else "", "gold_patch": v["patch"],
             "abs_test_patch": abs_suite.loc[iid, "test_patch"][:6000]}
    bad = [k for k, t in texts.items() if sha(t) != ix["text_sha256"][k]]
    if bad:
        raise SystemExit(f"{iid}: fetched {bad} differ from the text the paper's prompt used (sha256 mismatch)")
    return {"instance_id": iid, **texts, "failing_tests": ix["failing_tests"], "runs_read": ix["runs_read"],
            "first_errors": [], "agent_patches": [agent_patch(iid, a) for a in ix["agent_patches"]],
            "wbp_unique": ix["wbp_unique"]}


if RUN_GEMMA:
    verified, abs_suite = suite("verified"), suite("abs")
    packets = [build_packet(t, verified, abs_suite) for t in TASKS]
    for p in packets:
        print(f"{p['instance_id']}: prompt {len(audit_prompt.render(p)):,} characters, "
              f"sha256 matches for issue, hints, gold patch, SWE-ABS test patch and {len(p['agent_patches'])} agent patch(es)")

In [ ]:
def find_model_dir():
    # An attached Kaggle model (a folder with config.json whose path names E4B), else the Hugging Face id.
    if MODEL_OVERRIDE:
        return MODEL_OVERRIDE, None
    if Path("/kaggle/input").is_dir():
        for c in sorted(Path("/kaggle/input").glob("**/config.json")):
            if "e4b" in str(c.parent).lower() and "gemma" in str(c.parent).lower():
                return str(c.parent), None
    return E4B["spec"][3:], E4B["revision"]


def attn_map(config):
    # sdpa for the text model, eager for the unused audio and vision towers (E4B's audio tower rejects sdpa).
    subs = [k for k in getattr(config, "sub_configs", {}) if getattr(config, k, None) is not None]
    return {"": "sdpa", **{k: ("sdpa" if k == "text_config" else "eager") for k in subs}} if subs else "sdpa"


if RUN_GEMMA:
    if not MODEL_OVERRIDE:
        # The library versions of the paper's runs (Part 1 does not import transformers, so no restart is needed).
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.2", "accelerate==1.15.0"],
                       check=True)
    import torch
    import transformers
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    print("transformers", transformers.__version__)
    assert transformers.__version__ == "5.20.2", "the paper's runs used transformers 5.20.2"

    path, revision = find_model_dir()
    if revision and not os.environ.get("HF_TOKEN"):
        # Gemma weights from Hugging Face are gated: read the token from the Kaggle secret HF_TOKEN, if there is one.
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:  # noqa: BLE001
            print(f"no HF_TOKEN secret read ({type(e).__name__}); the download works only if the token is set otherwise")
    gpu = torch.cuda.is_available()
    t0 = time.perf_counter()
    kw = {"revision": revision} if revision else {}
    mcfg = AutoConfig.from_pretrained(path, **kw)
    tok = AutoTokenizer.from_pretrained(path, **kw)
    load_kw = {"dtype": torch.float16 if gpu else torch.float32, "attn_implementation": attn_map(mcfg), **kw}
    if gpu:
        load_kw["device_map"] = "auto"
        if torch.cuda.device_count() >= 2:
            load_kw["max_memory"] = {int(k): v for k, v in E4B["max_memory"].items()}
    model = AutoModelForCausalLM.from_pretrained(path, **load_kw).eval()
    print(f"model: {path if not path.startswith('/kaggle') else 'attached Kaggle model ' + Path(path).name}, "
          f"revision {revision or 'as attached (not pinned)'}")
    print(f"loaded {type(model).__name__} in {time.perf_counter() - t0:.0f} s on "
          f"{torch.cuda.device_count() if gpu else 0} GPU(s), torch {torch.__version__}")

In [ ]:
def audit(p: dict) -> dict:
    enc = tok.apply_chat_template([{"role": "user", "content": audit_prompt.render(p)}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    dev = next(q.device for q in model.parameters() if q.device.type != "meta")
    enc = {k: v.to(dev) for k, v in enc.items()}
    n = int(enc["input_ids"].shape[1])
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=cfg["max_new_tokens"], do_sample=False,
                             prefill_chunk_size=cfg["prefill_chunk"])
    secs = time.perf_counter() - t0
    raw = tok.decode(out[0, n:], skip_special_tokens=True)
    peak = ({i: torch.cuda.max_memory_allocated(i) / 2 ** 30 for i in range(torch.cuda.device_count())}
            if torch.cuda.is_available() else {})
    return {"instance_id": p["instance_id"], "prompt_tokens": n, "gen_tokens": int(out.shape[1] - n),
            "tail": raw.strip()[-300:], "verdict": audit_prompt.parse(raw), "seconds": secs, "peak_gib": peak}


if RUN_GEMMA:
    results = []
    for p in packets:
        r = audit(p)
        results.append(r)
        print(f"=== {r['instance_id']}  ({r['prompt_tokens']} prompt tokens, {r['gen_tokens']} generated)")
        print("last 300 characters of the answer (model text may quote the task):")
        print(r["tail"])
        print()
    with open(WORK / "gemma_live_verdicts.jsonl", "w", encoding="utf-8") as fh:
        for r in results:
            fh.write(json.dumps({k: v for k, v in r.items() if k != "tail"}) + "\n")   # no model text saved

In [ ]:
if RUN_GEMMA:
    def shown(v):
        # A missing VERDICT line counts as not coupled (genuine), as in the paper.
        if v is None:
            return "no VERDICT line -> genuine"
        return f"{v} -> {'coupled' if audit_prompt.is_coupled(v) else 'genuine'}"

    live = "Gemma 4 E4B, live" if not MODEL_OVERRIDE else f"test model {MODEL_OVERRIDE.split('/')[-1]}, live"
    table = pd.DataFrame([{
        "task": r["instance_id"],
        "reference (two readers)": labels[r["instance_id"]]["gold"],
        live: shown(r["verdict"]),
        "Gemma 4 E4B, saved A1 run": shown(saved[("e4b", r["instance_id"])]["verdict"]),
        "wall time (s)": round(r["seconds"], 1),
        "peak allocated GPU memory (GiB)": " + ".join(f"{g:.1f}" for g in r["peak_gib"].values()) or "CPU run",
    } for r in results])
    display(table)

**Reading the result.** A no-VERDICT answer counts as not coupled, as in the paper (whose runs also had a forced-choice
fallback that this demo leaves out). Agreement on two tasks is not evidence either way: the paper's estimate rests on
76 tasks, where E4B reached kappa 0.127 and the 12B 0.242, a high-precision but low-recall flagger (21 of its 28
coupled calls right, about half of the coupled tasks missed). What the demo does show is the cost of one local audit:
the wall time and peak memory above, on two free-tier GPUs.